# 05 — Classification (`case_type_grouped`)

Final W8 comparison of all four §3 models on the **5-class** task. Produces
the figures for slide 9:

- Macro-F1 / per-class precision-recall table across NB, LR, Bi-LSTM, BERT.
- One confusion matrix per model.
- Best-model per-class report for the slide bullet.

This notebook assumes you've already trained all four models — locally for
NB/LR (`python -m src.classify.train --task case_type --model {nb,lr}`) and on
Colab for Bi-LSTM/BERT (see `06_classification_colab.ipynb`). The Colab zip
should be unpacked under the project root so `models/` and `results/` line up.

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.utils import RESULTS_DIR

TASK = 'case_type'
TAG = 'casetype'
MODELS = ['nb', 'lr', 'lstm', 'bert']

## 1. Headline metrics across all 4 models

Filtered + pivoted from `results/classification_metrics.csv`.

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / 'classification_metrics.csv')
metrics = metrics.drop_duplicates(
    subset=['task', 'model', 'input_source', 'split'], keep='last'
)
test = metrics[(metrics.task == TASK) & (metrics.split == 'test')]
test = test.set_index('model').reindex(MODELS)
display(test[['accuracy', 'f1_macro', 'f1_weighted', 'auc_roc', 'train_seconds', 'notes']])

## 2. Bar chart: Macro-F1 by model (slide 9 headline)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(test.index, test['f1_macro'], color=['#888', '#1f77b4', '#2ca02c', '#d62728'])
for i, v in enumerate(test['f1_macro']):
    if not pd.isna(v):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center')
ax.set_ylabel('Macro F1 (test)')
ax.set_title('case_type · macro F1 by model')
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'case_type_macroF1_by_model.png', dpi=150)
plt.show()

## 3. Per-class precision/recall — load from JSON reports

In [ ]:
rows = []
for model in MODELS:
    p = RESULTS_DIR / 'classification_reports' / f'{model}_{TAG}_test.json'
    if not p.exists():
        print(f'⚠️ missing {p}; train {model} first.')
        continue
    report = json.loads(p.read_text())
    for cls_name, vals in report.items():
        if cls_name in {'accuracy', 'macro avg', 'weighted avg'}:
            continue
        rows.append({
            'model': model, 'class': cls_name,
            'precision': vals['precision'], 'recall': vals['recall'],
            'f1': vals['f1-score'], 'support': vals['support'],
        })
per_class = pd.DataFrame(rows)
per_class_f1 = per_class.pivot(index='class', columns='model', values='f1')
display(per_class_f1.round(3))

## 4. Confusion matrices side-by-side (one per model)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, model in zip(axes.ravel(), MODELS):
    img_path = RESULTS_DIR / 'confusion_matrices' / f'{model}_{TAG}_test.png'
    if not img_path.exists():
        ax.text(0.5, 0.5, f'{model.upper()}\n(not trained yet)', ha='center', va='center')
        ax.axis('off')
        continue
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'{model.upper()} · case_type')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'case_type_confusion_grid.png', dpi=120)
plt.show()

## 5. Training curves (LSTM + BERT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, model in zip(axes, ['lstm', 'bert']):
    img_path = RESULTS_DIR / 'training_curves' / f'{model}_{TAG}.png'
    if not img_path.exists():
        ax.text(0.5, 0.5, f'{model.upper()}\n(not trained yet)', ha='center', va='center')
        ax.axis('off')
        continue
    img = plt.imread(img_path)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'{model.upper()} training curves · case_type')
fig.tight_layout()
plt.show()

## 6. Slide-ready summary text

Auto-generated bullet points for slide 9.

In [ ]:
if test['f1_macro'].notna().any():
    best = test['f1_macro'].dropna().idxmax()
    print(f'- Best model: **{best.upper()}** (macro F1 = {test.loc[best, "f1_macro"]:.3f})')
    print(f'- Baseline gap: ΔF1 (best − NB) = {(test.loc[best, "f1_macro"] - test.loc["nb", "f1_macro"]):+.3f}')
    smallest = per_class.loc[per_class.groupby("class")["support"].first().idxmin()] if not per_class.empty else None
    if smallest is not None:
        print(f'- Smallest class (`{smallest.iloc[0]["class"]}`): see f1 by model in §3 table.')
else:
    print('Train deep models first (notebook 06) to populate this section.')